# SafeSound 기준선 학습 (v1)**셀을 위에서부터 순서대로 실행한다.** 셀 2는 런타임을 재시작시키므로, 재시작 후셀 3부터 이어서 누르면 된다.- 데이터: `dataset-v1` 태그 (창 13,007 / 5,207, int8 1.2초 저장)- 모델: KWS20 v3 파생 1D CNN, 5클래스, QAT 8bit- 산출물 1순위는 **혼동행렬**이다 — 배경음이 이벤트 합의 2.8배라 전체 정확도는  전부 배경음으로 찍어도 74%가 나온다> 런타임 유형을 **GPU (T4)** 로 먼저 바꿀 것. 수정 → 노트 설정 → 하드웨어 가속기.

## 0. Drive 에 올려둬야 하는 파일| 파일 | 내용 ||---|---|| `MyDrive/max78000/safesound-v1.tar.gz` + `.sha256` | 데이터셋 (140MB) || `MyDrive/max78000/safesound-code-v1.tar.gz` + `.sha256` | 우리 코드 || `MyDrive/max78000/ai8x-conda-env.tar.gz` *(선택)* | 환경 백업 3.2GB — 있으면 셀 3이 10분 아낀다 |없으면 셀 3이 환경을 처음부터 만든다 (약 15~20분).

In [ ]:
# 1. Drive 마운트 — 데이터·코드·체크포인트가 모두 여기에 있다from google.colab import drivedrive.mount('/content/drive')BASE = '/content/drive/MyDrive/max78000'import osfor f in ['safesound-v1.tar.gz', 'safesound-v1.tar.gz.sha256',          'safesound-code-v1.tar.gz', 'safesound-code-v1.tar.gz.sha256']:    print(('있음  ' if os.path.exists(f'{BASE}/{f}') else '없음  ') + f)print(('있음  ' if os.path.exists(f'{BASE}/ai8x-conda-env.tar.gz')       else '없음(처음부터 빌드)  ') + 'ai8x-conda-env.tar.gz')

In [ ]:
# 2. condacolab 설치 — ai8x 는 Python 3.11 을 요구하는데 Colab 기본은 3.13 이다 (런타임 재시작됨)!pip install -q condacolabimport condacolabcondacolab.install()

In [ ]:
# 3. ai8x 환경 준비 — Drive 백업이 있으면 복원, 없으면 CLAUDE.md 12장의 해결책을 적용해 새로 만든다import os, subprocess, sysBASE = '/content/drive/MyDrive/max78000'ENV = '/usr/local/envs/ai8x'def sh(cmd):    print('$', cmd)    r = subprocess.run(cmd, shell=True)    if r.returncode:        raise SystemExit(f'실패: {cmd}')# 어느 경로든 필요하다: torchaudio 의 speed_augment 가 시스템 libsox 를 쓴다.# conda 환경 밖이라 환경 백업에 포함되지 않는다 (CLAUDE.md 12장)sh('apt-get -qq install -y libsox3 libsox-dev libsox-fmt-all sox')os.environ['MPLBACKEND'] = 'Agg'   # Colab 의 matplotlib_inline 백엔드가 conda 환경에 없다if os.path.exists(f'{BASE}/ai8x-conda-env.tar.gz') and not os.path.exists(ENV):    print('Drive 백업에서 복원한다 (약 3분)')    sh(f'mkdir -p {ENV} && tar xzf {BASE}/ai8x-conda-env.tar.gz -C /usr/local/envs')elif not os.path.exists(ENV):    print('환경을 새로 만든다 (약 15~20분)')    sh('conda create -y -n ai8x python=3.11')    # torch 2.3.1 은 PyPI 에서 내려갔다 → PyTorch 공식 cu121 인덱스에서 받는다    sh('conda run -n ai8x --no-capture-output pip install -q '       'torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121')else:    print('환경이 이미 있다')print('\n환경 확인:')sh('conda run -n ai8x --no-capture-output python -c '   '"import sys, torch; print(sys.version.split()[0], torch.__version__, '   'torch.cuda.is_available())"')

In [ ]:
# 4. ai8x-training 을 받고 CLAUDE.md 12장의 패치를 적용한다 (distiller 충돌·예외 처리)import os, subprocessdef sh(cmd):    print('$', cmd)    r = subprocess.run(cmd, shell=True)    if r.returncode:        raise SystemExit(f'실패: {cmd}')if not os.path.exists('/content/ai8x-training'):    sh('git clone -q --recursive https://github.com/analogdevicesinc/ai8x-training.git '       '/content/ai8x-training')os.chdir('/content/ai8x-training')# (1) distiller 디렉터리명이 네임스페이스 패키지 충돌을 만든다 —#     /content/ai8x-training/distiller 가 설치된 패키지를 가린다 → 이름을 바꾼다if os.path.isdir('distiller') and not os.path.isdir('distiller_repo'):    sh('mv distiller distiller_repo')# (2) distiller/__init__.py 의 except pkg_resources.DistributionNotFound 는#     ContextualVersionConflict 를 못 잡아 모듈 로딩이 중간에 끊긴다.#     knowledge_distillation 없음 / __file__ None / __version__ 없음 증상의 공통 원인init = 'distiller_repo/distiller/__init__.py'if os.path.exists(init):    s = open(init, encoding='utf-8').read()    if 'except pkg_resources.DistributionNotFound' in s:        open(init, 'w', encoding='utf-8').write(            s.replace('except pkg_resources.DistributionNotFound', 'except Exception'))        print('distiller __init__ 예외 처리 패치 적용')sh('conda run -n ai8x --no-capture-output pip install -q -r requirements.txt || true')sh('conda run -n ai8x --no-capture-output pip install -q -e distiller_repo || true')# 로그·체크포인트는 Drive 에 둔다 — 세션이 끊겨도 남고, 재실행 시 여기서 재개한다sh('rm -rf logs && mkdir -p /content/drive/MyDrive/max78000/logs && '   'ln -s /content/drive/MyDrive/max78000/logs logs')print('\nlogs ->', os.path.realpath('logs'))

In [ ]:
# 5. 우리 코드 tar 를 해시 확인 후 풀고, ai8x 쪽에 심볼릭 링크를 건다import hashlib, os, subprocess, tarfileBASE = '/content/drive/MyDrive/max78000'def verify(path):    want = open(path + '.sha256').read().split()[0]    h = hashlib.sha256()    with open(path, 'rb') as f:        for c in iter(lambda: f.read(1 << 20), b''):            h.update(c)    got = h.hexdigest()    print(f'{os.path.basename(path)}  {"OK" if got == want else "불일치!"}')    assert got == want, f'{path} 손상 — 다시 올릴 것'    return gotverify(f'{BASE}/safesound-code-v1.tar.gz')os.makedirs('/content/max78000-proj', exist_ok=True)with tarfile.open(f'{BASE}/safesound-code-v1.tar.gz') as tf:    tf.extractall('/content/max78000-proj')# 벤더 레포는 고치지 않고 링크만 건다 (CLAUDE.md 11장)for src, dst in [('datasets/safesound.py', 'datasets/safesound.py'),                 ('models/ai85net-safesound.py', 'models/ai85net-safesound.py')]:    d = f'/content/ai8x-training/{dst}'    if os.path.islink(d) or os.path.exists(d):        os.remove(d)    os.symlink(f'/content/max78000-proj/{src}', d)    print('link', d, '->', os.readlink(d))import jsonprint('\n코드 번들:', json.load(open('/content/max78000-proj/CODE_MANIFEST.json'))['commit'][:9])

In [ ]:
# 6. 데이터셋 해시 확인 후 압축 해제 — 경로는 data/SafeSound/{train,test}/{클래스}/ 가 된다import hashlib, json, os, tarfileBASE = '/content/drive/MyDrive/max78000'DATA = '/content/ai8x-training/data/SafeSound'want = open(f'{BASE}/safesound-v1.tar.gz.sha256').read().split()[0]h = hashlib.sha256()with open(f'{BASE}/safesound-v1.tar.gz', 'rb') as f:    for c in iter(lambda: f.read(1 << 20), b''):        h.update(c)assert h.hexdigest() == want, '데이터 tar 손상 — 다시 올릴 것'print('sha256 OK', want[:16], '...')os.makedirs(DATA, exist_ok=True)with tarfile.open(f'{BASE}/safesound-v1.tar.gz') as tf:    tf.extractall(DATA)m = json.load(open(f'{DATA}/MANIFEST.json'))print('\n데이터셋', m['tag'], '커밋', m['commit'][:9])for k, v in sorted(m['counts'].items()):    print(f'  {k:<22}{v:>7}창')print('  합계', sum(m['counts'].values()))

In [ ]:
# 7. 학습 전 필수 검증 — KAT(축·스케일)과 클래스 가중치. 여기서 실패하면 학습하지 말 것!cd /content/max78000-proj && conda run -n ai8x --no-capture-output python tools/kat_safesound.py!conda run -n ai8x --no-capture-output python -c "\import sys; sys.path.insert(0,'/content/ai8x-training'); sys.path.insert(0,'/content/max78000-proj/datasets');\import safesound as S;\print('클래스 가중치(역빈도):', dict(zip(S.CLASSES, S.datasets[0]['weight'])));\print('실측 재계산     :', dict(zip(S.CLASSES, S.class_weights('/content/ai8x-training/data/SafeSound'))))"

In [ ]:
# 8. 기준선 학습 — 재실행하면 마지막 체크포인트에서 자동으로 이어서 간다import glob, os, subprocessos.chdir('/content/ai8x-training')os.environ['MPLBACKEND'] = 'Agg'PROJ = '/content/max78000-proj'# 가장 최근 실행 폴더의 체크포인트를 찾는다 (distiller 는 logs/<name>___<타임스탬프>/ 를 만든다)ckpts = sorted(glob.glob('logs/safesound-v1___*/*checkpoint.pth.tar'), key=os.path.getmtime)resume = f'--resume-from {ckpts[-1]}' if ckpts else ''print('재개:', ckpts[-1] if ckpts else '(없음 — 처음부터)')cmd = f'''conda run -n ai8x --no-capture-output python train.py \  --epochs 150 --optimizer Adam --lr 0.001 --wd 0 --deterministic --seed 1 \  --compress {PROJ}/colab/schedule_safesound.yaml \  --qat-policy {PROJ}/colab/qat_policy_safesound.yaml \  --model ai85safesoundnet --dataset SafeSound --device MAX78000 \  --confusion --batch-size 128 --workers 2 \  --name safesound-v1 --out-dir logs {resume}'''print(cmd)subprocess.run(cmd, shell=True)

In [ ]:
# 9. 평가 — 창/원본 혼동행렬, 원본 부트스트랩 CI, 시간당 오경보. 결과를 Drive 에 텍스트로 남긴다import glob, os, subprocess, timeos.chdir('/content/ai8x-training')BASE = '/content/drive/MyDrive/max78000'best = sorted(glob.glob('logs/safesound-v1___*/*best*.pth.tar'), key=os.path.getmtime)assert best, 'best 체크포인트가 없다 — 셀 8을 먼저 끝낼 것'print('평가 대상:', best[-1])out = f'{BASE}/eval-{time.strftime("%Y%m%d-%H%M%S")}.txt'cmd = (f'conda run -n ai8x --no-capture-output python '       f'/content/max78000-proj/tools/eval_confusion.py '       f'--checkpoint {best[-1]} --data /content/ai8x-training/data '       f'--ai8x /content/ai8x-training --hop-ms 250')r = subprocess.run(cmd, shell=True, capture_output=True, text=True)text = r.stdout + r.stderrprint(text)open(out, 'w', encoding='utf-8').write(text)print('\n저장:', out)

## 10. 학습이 끝난 뒤 (WSL2 에서)혼동행렬의 **N프레임 오경보는 하한(낙관치)** 이라 논문에 쓰지 않는다. 논문 수치는원본 오디오를 빈틈없이 훑는 스트리밍 평가에서 나오고, 그건 원본이 있는 WSL2 에서돌리는 편이 간단하다.```bash# WSL2 — best 체크포인트를 Drive 에서 내려받은 뒤~/ai8x-training/venv/bin/python tools/stream_eval.py \  --checkpoint <best.pth.tar> --ai8x ~/ai8x-training \  --json docs/results/stream-eval-pc.json```그다음 양자화·합성으로 넘어간다 (442KB / 512KB 초과 여부 확인).**읽는 순서**: 혼동행렬(원본 단위) → 클래스별 재현율·CI → 시간당 오경보.전체 정확도는 보지 않는다.